# IBKR API notebook

#### Connection

In [ ]:
from ib_async import *
import pandas as pd
import logging
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

## Request Historical data

#### Choose your contract

In [ ]:
contract = Index('NDX', 'NASDAQ', 'USD')
print("Contract details:")
print("Symbol:", contract.symbol)
print("Exchange:", contract.exchange)
print("Currency:", contract.currency)

#### Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
print(f"First date of data available: {formatted_time}")

#### Request historical data function

In [ ]:
save_path = "./database/AAPL_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
print(f"First date in the DataFrame: {first_date}")
print(f"End date: {end_date}")

In [ ]:
#today's date
end_date = pd.Timestamp.now().strftime('%Y%m%d %H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [ ]:
historical_data_interval = '10 secs' 
request_duration = '5 Y'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow='TRADES',
        useRTH=False,
        formatDate=2,
        timeout = 0)

In [ ]:
bars[0]

Convert the list of bars to a data frame and print the first and last rows:

In [ ]:
df = util.df(bars)
print("DataFrame shape:", df.shape)

display(df.head(n=20))
display(df.tail(n=20))

Save your pulled data in a dataframe

Compression possibilities sorted by compression ratio from the lowest to the highest : 
- `snappy`

- `gzip`

- `brotli`

#### Checking if volume and average columns are empty or not and remove it if empty

In [ ]:
# Check if the 'volume' column is empty (all values are 0.0)
if (df['volume'] == 0.0).all():
    df = df.drop(columns=['volume'])  # Drop the 'volume' column
    print("The 'volume' column was empty and has been removed.")

# Check if the 'average' column is empty (all values are 0.0)
if (df['average'] == 0.0).all():
    df = df.drop(columns=['average'])  # Drop the 'average' column
    print("The 'average' column was empty and has been removed.")

# Display the updated DataFrame
display(df.head(n=30))

Construction du nom du fichier et sauvegarde en `.parquet` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate.parquet`

In [ ]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
start_date = df['date'].iloc[0].strftime('%Y%m%d')
end_date = df['date'].iloc[-1].strftime('%Y%m%d')
# Structure
save_path = f"../marketData/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

In [ ]:
import pandas as pd
save_path = "../marketData/NDX_10secs_20220214_to_20250411.parquet"
#save_path = "../marketData/NDX_10secs_20220131_to_20250403.parquet"

# Load the parquet file into a DataFrame

retrieved_df = pd.read_parquet(save_path)
retrieved_df['date'] = retrieved_df['date'].dt.tz_convert('Europe/Paris')
print(type(retrieved_df['date'].iloc[0]))



# Display the first and last rows of the DataFrame
display(retrieved_df.head())
display(retrieved_df.tail())

Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Plot the close data

In [ ]:
df.plot(y='close');

There is also a utility function to plot bars as a candlestick plot. It can accept either a DataFrame or a list of bars. Here it will print the last 100 bars:

In [ ]:
util.barplot(bars[-100:], title=contract.symbol);

## Historical data with realtime updates

A new feature of the API is to get live updates for historical bars. This is done by setting `endDateTime` to an empty string and the `keepUpToDate` parameter to `True`.

Let's get some bars with an keepUpToDate subscription:

In [ ]:
bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='900 S',
        barSizeSetting='10 secs',
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1,
        keepUpToDate=True)

Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(10)
ib.cancelHistoricalData(bars)

Realtime bars
------------------

With ``reqRealTimeBars`` a subscription is started that sends a new bar every 5 seconds.

First we'll set up a event handler for bar updates:

In [ ]:
def onBarUpdate(bars, hasNewBar):
    print(bars[-1])

Then do the real request and connect the event handler,

In [ ]:
bars = ib.reqRealTimeBars(contract, 5, 'MIDPOINT', False)
bars.updateEvent += onBarUpdate

let it run for half a minute and then cancel the realtime bars.

In [ ]:
ib.sleep(30)
ib.cancelRealTimeBars(bars)

The advantage of reqRealTimeBars is that it behaves more robust when the connection to the IB server farms is interrupted. After the connection is restored, the bars from during the network outage will be backfilled and the live bars will resume.

reqHistoricalData + keepUpToDate will, at the moment of writing, leave the whole API inoperable after a network interruption.

### Request historical market news

In [ ]:
news_providers = ib.reqNewsProviders()
print("News Providers:", news_providers)
for provider in news_providers:
    print(f"Code: {provider.code}, Name: {provider.name}")

In [ ]:
article = ib.reqNewsArticle(providerCode="BRFG")
print(article)

In [ ]:
historical_news = ib.reqHistoricalNews(
    conId=contract.conId,  # ID du contrat
    providerCodes="BRFG",
    startDateTime="20250401 00:00:00",
    endDateTime="20250405 23:59:59",
    totalResults=10
)
for news in historical_news:
    print(f"Date: {news.time}, Title: {news.headline}")

In [ ]:
newsbulletin = ib.reqNewsBulletins(allMessages=True)

In [ ]:
ib.disconnect()

## Additional features

#### Data pre processing

In [1]:
import os
import pandas as pd
from igtrader.Strategies.Helpers import load_data, resample_ohlc

In [3]:
symbol = 'NDX'
interval = '10secs'
start_date = '20220214'
end_date = '20250411'

save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}.parquet"
df = pd.read_parquet(save_path, engine='pyarrow')
df

,date,open,high,low,close
0,2022-02-14 14:30:10+00:00,14233.40,14239.47,14233.40,14239.47
1,2022-02-14 14:30:20+00:00,14239.47,14242.25,14231.47,14235.56
2,2022-02-14 14:30:30+00:00,14235.56,14242.16,14235.56,14239.24
3,2022-02-14 14:30:40+00:00,14239.24,14239.45,14233.44,14236.41
4,2022-02-14 14:30:50+00:00,14236.41,14257.50,14236.41,14257.50
...,...,...,...,...,...
1848343,2025-04-11 19:59:10+00:00,18669.67,18673.95,18664.38,18673.95
1848344,2025-04-11 19:59:20+00:00,18673.69,18683.45,18672.71,18683.45
1848345,2025-04-11 19:59:30+00:00,18679.30,18681.99,18675.94,18677.13
1848346,2025-04-11 19:59:40+00:00,18674.72,18676.22,18671.43,18671.43


In [ ]:
interval= 'secs'
df = resample_ohlc(df, interval)
df

,open,high,low,close
date,,,,
2022-02-14 14:30:30+00:00,14239.24,14257.50,14233.44,14257.50
2022-02-14 14:31:00+00:00,14257.50,14267.11,14256.63,14261.01
2022-02-14 14:31:30+00:00,14261.01,14261.01,14238.47,14238.47
2022-02-14 14:32:00+00:00,14238.47,14238.47,14228.55,14228.79
2022-02-14 14:32:30+00:00,14228.79,14230.59,14209.07,14209.81
...,...,...,...,...
2025-04-11 19:57:30+00:00,18688.54,18690.88,18686.66,18690.88
2025-04-11 19:58:00+00:00,18691.44,18694.39,18681.51,18681.77
2025-04-11 19:58:30+00:00,18681.75,18681.75,18673.67,18675.68


In [7]:
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}.parquet"
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

Fichier sauvegardé sous le nom : ../marketData/NDX_30secs_20220214_to_20250411.parquet


In [ ]:
# Remove the 'barCount' column from the DataFrame
df = df.drop(columns=['barCount'])

# Display the updated DataFrame
display(df.head())
df.to_parquet(save_path, index=True, compression=None)